# Calendar: Data Overview

Each row describes the availability of a listing on a specific calendar date. Together, `listing_id` and `date` form the unique key.

| Feature | Description | Decision |
|---|---|---|
| `listing_id` | Listing ID; links to `listings.id` | Keep |
| `date` | Calendar date (YYYY-MM-DD) | Keep |
| `available` | `t`: available; `f`: unavailable | Keep |
| `minimum_nights` | Minimum length of stay in nights | Drop |
| `maximum_nights` | Maximum length of stay in nights | Drop |

**Feature removal:** minimum_nights, maximum_nights as this info is already contained in the listings.csv.
An unavailable day does not necessarily mean that the listing is booked.


## Load the File



In [36]:
import pandas as pd

from utils.constants import DATA_PATH

AIRBNB_DATA = DATA_PATH / "airbnb"
PROCESSED_DATA = AIRBNB_DATA / "processed"

FILE_PATH = AIRBNB_DATA / "calendar.csv"

df = pd.read_csv(FILE_PATH)


## 1. Preview the First Rows

In [37]:
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())

Rows: 1,754,190 | Columns: 5


,listing_id,date,available,minimum_nights,maximum_nights
0,145320,2026-06-30,f,2,1125
1,145320,2026-07-01,f,2,1125
2,145320,2026-07-02,f,2,1125
3,145320,2026-07-03,f,2,1125
4,145320,2026-07-04,f,2,1125


## 2. Inspect Data Types and Missing Values

In [38]:
display(pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Missing values": df.isna().sum(),
    "Unique values": df.nunique(),
}))

,Data type,Missing values,Unique values
listing_id,int64,0,4806
date,str,0,368
available,str,0,2
minimum_nights,int64,0,65
maximum_nights,int64,0,158


## 3. Drop columns, convert columns

In [39]:
df.drop(columns=['minimum_nights', 'maximum_nights'], inplace=True)

# Make 'date' column a datetime object
df['date'] = pd.to_datetime(df['date'])

In [40]:
df.dtypes

listing_id             int64
date          datetime64[us]
available                str
dtype: object

# 4. Remove non-existing ids

Since some rows may have been removed in the processed listings.csv, we should check which listing_ids are contained in that file.

In [41]:
LISTINGS = PROCESSED_DATA / "processed_listings_data.csv"
assert LISTINGS.exists(), f"File not found: {LISTINGS}"
listings_df = pd.read_csv(LISTINGS)

In [42]:
# Check which listing ids in calendar are also in the listings dataframe
calendar_listing_ids = df['listing_id'].unique()
listings_listing_ids = listings_df['id'].unique()

# Retrieve the listing ids in calendar that does not exist in listings
missing_listing_ids = set(calendar_listing_ids) - set(listings_listing_ids)
print(f"Missing listing ids in calendar that are not in listings:\n{len(missing_listing_ids)}")

Missing listing ids in calendar that are not in listings:
0


In [43]:
# Remove the rows in the calendar dataframe that have listing ids not present in the listings dataframe
print(f"Number of rows before removing missing listing ids: {len(df)}")
df = df[df['listing_id'].isin(listings_listing_ids)]
print(f"Number of rows after removing missing listing ids: {len(df)}")

Number of rows before removing missing listing ids: 1754190
Number of rows after removing missing listing ids: 1754190


## Export the CSV


In [44]:
# Raw data can be used as nothing has changed
df.to_csv(PROCESSED_DATA / "processed_calendar_data.csv", index=False)